In [ ]:
from pathlib import Path
import subprocess
import sys
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(project_root / 'requirements.txt')])


In [ ]:
from pathlib import Path
import os
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MODELS_DIR = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
print(f'Seed fixed at {SEED}')
print(f'Project root: {PROJECT_ROOT}')


# Notebook 03 ? Multi-Hazard Mask Preparation
## Rasterizes and inventories aligned flood, erosion, and landslide labels


## Section 3.1 — Load UNOSAT Shapefile

Inspecting the source polygons before rasterization is essential because CRS mismatches or empty geometries would propagate directly into the training labels. This section also records the flood-area benchmark used later in validation.


In [ ]:
import re
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize

RAW_UNOSAT = RAW_DIR / 'unosat'
RAW_CEMS = RAW_DIR / 'cems'
RAW_INFRA = RAW_DIR / 'infrastructure'
PROCESSED_SAR = PROCESSED_DIR / 'sar'
PROCESSED_TERRAIN = PROCESSED_DIR / 'terrain'
PROCESSED_MASKS = PROCESSED_DIR / 'masks'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
REPORT_DIR = OUTPUTS_DIR / 'report'
PROCESSED_MASKS.mkdir(parents=True, exist_ok=True)

label_files = sorted(list(RAW_UNOSAT.rglob('*.shp')) + list(RAW_CEMS.rglob('*.shp')))
if not label_files:
    raise FileNotFoundError('No label shapefiles found under data/raw/unosat or data/raw/cems/.')
admin_candidates = list(RAW_INFRA.rglob('*.shp'))
admin_gdf = gpd.read_file(admin_candidates[0]).to_crs(32646) if admin_candidates else None
label_layers = []
for shp in label_files:
    gdf = gpd.read_file(shp).to_crs(32646)
    match = re.search(r'(20\d{2})(\d{2})(\d{2})', shp.stem)
    year_match = re.search(r'(20\d{2})', shp.stem)
    label_layers.append({'source_path': str(shp), 'source_name': shp.stem, 'date_token': match.group(0) if match else None, 'year_token': year_match.group(1) if year_match else None, 'gdf': gdf, 'area_km2': gdf.geometry.area.sum() / 1e6, 'polygon_count': len(gdf)})
label_summary_df = pd.DataFrame([{k: v for k, v in row.items() if k != 'gdf'} for row in label_layers])
display(label_summary_df[['source_name', 'date_token', 'year_token', 'polygon_count', 'area_km2']])
focus_layer = label_layers[0]
print(f"Total flooded area km²: {focus_layer['area_km2']:,.2f}")
print(f"Polygon count: {focus_layer['polygon_count']:,}")
print(f"Detection tag: {focus_layer['date_token'] or focus_layer['year_token'] or 'multi-date'}")
ax = focus_layer['gdf'].plot(figsize=(8, 8), color='#2d6a4f', alpha=0.7, edgecolor='black')
if admin_gdf is not None:
    admin_gdf.boundary.plot(ax=ax, color='firebrick', linewidth=0.6)
plt.tight_layout(); plt.show()


## Section 3.2 — Rasterize to SAR Grid

Rasterization must use the exact SAR transform, resolution, and CRS to avoid label drift. This alignment is critical for both model training and scene-level validation later in the workflow.


In [ ]:
def rasterize_mask(shapefile_gdf, reference_tif, output_path):
    with rasterio.open(reference_tif) as ref:
        arr = rasterize([(geom, 1) for geom in shapefile_gdf.geometry if geom is not None and not geom.is_empty], out_shape=(ref.height, ref.width), transform=ref.transform, fill=0, dtype='uint8')
        profile = ref.profile.copy(); profile.update(count=1, dtype='uint8', nodata=0)
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(arr, 1)
    return output_path

def matching_vv_files(layer):
    vv_files = sorted(PROCESSED_SAR.glob('S1_VV_*.tif'))
    if layer['date_token']:
        exact = [p for p in vv_files if p.stem.endswith(layer['date_token'])]
        if exact:
            return exact
    if layer['year_token']:
        year_hits = [p for p in vv_files if p.stem.split('_')[-1].startswith(layer['year_token'])]
        if year_hits:
            return year_hits
    return vv_files

mask_records = []
for layer in label_layers:
    for vv_ref in matching_vv_files(layer):
        date_token = vv_ref.stem.split('_')[-1]
        out_mask = PROCESSED_MASKS / f"mask_{layer['source_name']}_{date_token}.tif"
        rasterize_mask(layer['gdf'], vv_ref, out_mask)
        mask_records.append({'label_source': layer['source_name'], 'date': date_token, 'year': date_token[:4], 'mask_path': str(out_mask), 'reference_vv': str(vv_ref)})
mask_manifest_df = pd.DataFrame(mask_records)
mask_manifest_df.to_csv(REPORT_DIR / 'mask_manifest.csv', index=False)
display(mask_manifest_df.head(20))


## Section 3.3 — Apply Permanent Water Mask

Removing permanent water helps prevent the model from learning stable haor and river water as flood. This is especially important in Sylhet where permanent and transient water are easily confused in SAR imagery.


In [ ]:
jrc_path = PROCESSED_TERRAIN / 'jrc_water_sylhet.tif'
if not jrc_path.exists():
    raise FileNotFoundError('Aligned JRC permanent water raster not found. Run Notebook 02 first.')

pixel_rows = []
with rasterio.open(jrc_path) as jrc_src:
    jrc = jrc_src.read(1).astype(bool)
for row in mask_records:
    mask_path = Path(row['mask_path'])
    with rasterio.open(mask_path) as src:
        arr = src.read(1).astype(np.uint8)
        profile = src.profile.copy()
    before = int(arr.sum())
    arr[jrc] = 0
    after = int(arr.sum())
    with rasterio.open(mask_path, 'w', **profile) as dst:
        dst.write(arr, 1)
    pixel_rows.append({'date': row['date'], 'flood_pixels_before': before, 'flood_pixels_after': after})
pixel_df = pd.DataFrame(pixel_rows)
display(pixel_df)


## Section 3.4 — Class Balance Check

Flood labels are usually highly imbalanced, so a quick balance table and chart help justify the loss design before training begins. This is also useful for spotting dates that may need more careful sampling attention.


In [ ]:
class_rows = []
for row in mask_records:
    with rasterio.open(row['mask_path']) as src:
        mask_arr = src.read(1)
    flood_pixels = int(mask_arr.sum())
    total_pixels = int(mask_arr.size)
    non_flood_pixels = total_pixels - flood_pixels
    class_rows.append({'date': row['date'], 'flood_pixels': flood_pixels, 'non_flood_pixels': non_flood_pixels, 'flood_pct': flood_pixels / total_pixels, 'imbalance_ratio': non_flood_pixels / max(flood_pixels, 1)})
class_balance_df = pd.DataFrame(class_rows)
display(class_balance_df)
fig, ax = plt.subplots(figsize=(10, 4), dpi=300)
ax.bar(class_balance_df['date'], class_balance_df['flood_pct'] * 100, color='#1d3557')
ax.axhline(10, color='firebrick', linestyle='--', label='1:10 threshold')
ax.set_ylabel('Flood pixels (%)')
ax.set_title('Mask class balance by date')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mask_class_balance.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)
if (class_balance_df['imbalance_ratio'] > 10).any():
    print('Note: imbalance > 1:10 detected; BCEDiceLoss will handle this during training.')


## Section 3.5 — Mask QC Visualization

Overlaying the cleaned masks on the SAR imagery confirms that the rasterized labels still track the flood footprint after permanent-water removal. This is the last checkpoint before patch extraction.


In [ ]:
for row in mask_records[: min(len(mask_records), 12)]:
    date_token = row['date']
    vv_path = Path(row['reference_vv'])
    mask_path = Path(row['mask_path'])
    with rasterio.open(vv_path) as vv_src, rasterio.open(mask_path) as mask_src:
        vv = vv_src.read(1); mask_arr = mask_src.read(1); extent = [vv_src.bounds.left, vv_src.bounds.right, vv_src.bounds.bottom, vv_src.bounds.top]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=300)
    axes[0].imshow(vv, cmap='gray', vmin=-25, vmax=0, extent=extent); axes[0].set_title(f"SAR VV {date_token}")
    axes[1].imshow(mask_arr, cmap='Reds', extent=extent); axes[1].set_title(f"Mask {row['label_source']}")
    axes[2].imshow(vv, cmap='gray', vmin=-25, vmax=0, extent=extent); axes[2].imshow(np.ma.masked_where(mask_arr == 0, mask_arr), cmap='autumn', alpha=0.6, extent=extent); axes[2].set_title('Overlay')
    plt.tight_layout(); plt.savefig(FIGURES_DIR / f"mask_qc_{row['label_source']}_{date_token}.png", dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)


<!-- MULTI-HAZARD EXTENSION GENERATED -->
## Section 3.6 ? Multi-Hazard Label Inventory
A true multi-hazard pipeline needs one aligned target layer per hazard. This section inventories flood, erosion, and landslide labels, then plans aligned raster outputs under `data/processed/masks_multihazard/`.


In [ ]:
# MULTI-HAZARD EXTENSION GENERATED
import pandas as pd
from analysis.multi_hazard_support import load_hazard_catalog, ensure_multihazard_tree

ROOT = Path(r'f:\MAPATHON\sylhet_flood_2024')
hazard_catalog = load_hazard_catalog(ROOT / 'config' / 'hazard_catalog.json')
paths = ensure_multihazard_tree(ROOT, hazard_catalog)

search_patterns = {
    'flood': [ROOT / 'data' / 'raw' / 'unosat', ROOT / 'data' / 'raw' / 'unosat_multi_year'],
    'erosion': [paths['raw_hazards'] / 'erosion' / 'labels'],
    'landslide': [paths['raw_hazards'] / 'landslide' / 'labels'],
}

rows = []
for hazard_id, folders in search_patterns.items():
    for folder in folders:
        if folder.exists():
            for shp in sorted(folder.rglob('*.shp')):
                rows.append({
                    'hazard': hazard_id,
                    'source_label': str(shp),
                    'planned_mask_output': str(paths['processed_masks'] / hazard_id / f"{shp.stem}.tif"),
                })
multi_hazard_label_inventory = pd.DataFrame(rows)
multi_hazard_label_inventory.to_csv(ROOT / 'outputs' / 'report' / 'multi_hazard_label_inventory.csv', index=False)
multi_hazard_label_inventory.head(20)
